In [1]:
#Load Clean Data
import pandas as pd
import numpy as np

df=pd.read_csv("clean_transactions_v1.csv")

df["transaction_datetime"]=pd.to_datetime(df["transaction_datetime"])

df=df.sort_values(["customer_id","transaction_datetime"])

print("shape:",df.shape)
print("fraud_rate_pct:",round(df["fraud_label"].mean()*100,2))

shape: (5000, 19)
fraud_rate_pct: 23.42


In [16]:
# FEATURE BUCKET 1 — Customer Behavior Features

#Transaction count last 1 hour / 24 hours
df["txn_count_1h"]=df.groupby("customer_id")["transaction_datetime"]\
    .transform(lambda x: x.diff().dt.total_seconds().fillna(0))

df["txn_count_1h"]=np.where(df["txn_count_1h"]<=3600,1,0)

df["txn_count_24h"]=df.groupby("customer_id")["transaction_datetime"]\
    .transform(lambda x: x.diff().dt.total_seconds().fillna(0))

df["txn_count_24h"]=np.where(df["txn_count_24h"]<=86400,1,0)

print(df[["txn_count_1h","txn_count_24h"]].head())

# Rolling 7-day average amount
df["avg_amount_7d"]=df.groupby("customer_id")["transaction_amount"]\
    .transform(lambda x: x.rolling(window=7,min_periods=1).mean())

# Max amount in last 24 transactions
df["max_amount_24h"]=df.groupby("customer_id")["transaction_amount"]\
    .transform(lambda x: x.rolling(window=5,min_periods=1).max())

   txn_count_1h  txn_count_24h
0             1              1
1             1              1
2             1              1
3             1              1
4             1              1


In [17]:
# FEATURE BUCKET 2 — Transaction Context
# Hour of transaction
df["txn_hour"]=df["transaction_datetime"].dt.hour

# Day of week
df["txn_day"]=df["transaction_datetime"].dt.dayofweek

# Weekend flag
df["is_weekend"]=np.where(df["txn_day"]>=5,1,0)

In [12]:
# FEATURE BUCKET 3 — Merchant Risk
# computing historical fraud rate per merchant_category.
merchant_risk=df.groupby("merchant_category")["fraud_label"].mean().reset_index()
merchant_risk.columns=["merchant_category","merchant_fraud_rate"]

df=df.merge(merchant_risk,on="merchant_category",how="left")

print(df[["merchant_category","merchant_fraud_rate"]].head())

  merchant_category  merchant_fraud_rate
0            Travel             0.235380
1           Grocery             0.251739
2            Luxury             0.195343
3            Luxury             0.195343
4           Grocery             0.251739


In [18]:
# FEATURE BUCKET 4 — Device & Geo Features

# New device flag (first time device used by customer)
df["new_device_flag"]=df.groupby("customer_id")["device_id"]\
    .transform(lambda x: ~x.duplicated()).astype(int)
#Geo switch flag
df["geo_switch_flag"]=df.groupby("customer_id")["customer_region"]\
    .transform(lambda x: x != x.shift(1)).astype(int)

In [19]:
# FEATURE BUCKET 5 — Velocity Features

#Time since last transaction
df["time_since_last_txn"]=df.groupby("customer_id")["transaction_datetime"]\
    .diff().dt.total_seconds().fillna(0)

#Count last 10 minutes
df["count_last_10min"]=np.where(df["time_since_last_txn"]<=600,1,0)

In [20]:
# Final Feature Matrix
feature_df=df.copy()

drop_cols=["transaction_id","customer_id","merchant_id","device_id","ip_address"]

feature_df=feature_df.drop(columns=[col for col in drop_cols if col in feature_df.columns])

print("Final feature matrix shape:",feature_df.shape)

Final feature matrix shape: (5000, 26)


In [21]:
# saving file
feature_df.to_csv("fraud_feature_matrix_v1.csv",index=False)
print("fraud_feature_matrix_v1.csv saved")

fraud_feature_matrix_v1.csv saved
